# YOLO + OCR Plate Detection - Local repo on Colab

This notebook is designed to run after cloning the repository locally in Colab.

- the YOLO model is loaded from the local repo
- the pipeline is imported from `src/plate_ocr`
- the image or folder is processed locally in the Colab runtime
- the results are saved in the local project folder

Important: do not re-download the project from GitHub on every run; clone it once in the Colab runtime and then run this notebook.

In [ ]:
# Cell 1 - Install dependencies
# Run this cell when starting the notebook.

%pip install -q ultralytics easyocr opencv-python-headless numpy pandas

print('Dependencies installed.')

In [ ]:
# Cell 2 - Local project path in Colab
# The repository should already be cloned inside /content before running this notebook.
# If it is not present, clone it once manually in Colab and then run the rest of the cells.

from pathlib import Path

WORKDIR = Path('/content')
PROJECT_ROOT = WORKDIR / 'AI-UAV-Address-Verification'

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f"The repository is not present in the Colab runtime: {PROJECT_ROOT}\n"
        "Clone it once in Colab with:\n"
        "!git clone https://github.com/senisama/AI-UAV-Address-Verification.git"
    )

%cd {PROJECT_ROOT}
print(f"Project root: {PROJECT_ROOT}")
print('Directory tree:')
!ls -la

In [ ]:
# Cell 3 - Define local paths from the repo

from pathlib import Path

PROJECT_ROOT = Path('/content/AI-UAV-Address-Verification')
MODEL_PATH = PROJECT_ROOT / 'models' / 'best.pt'
TEST_IMAGES_DIR = PROJECT_ROOT / 'data' / 'processed'
IMAGE_PATH = PROJECT_ROOT / 'uploaded_image.jpg'
OUTPUT_DIR = PROJECT_ROOT / 'outputs'
OUTPUT_DIR.mkdir(exist_ok=True)

print('PROJECT_ROOT =', PROJECT_ROOT)
print('MODEL_PATH =', MODEL_PATH)
print('TEST_IMAGES_DIR =', TEST_IMAGES_DIR)
print('IMAGE_PATH =', IMAGE_PATH)
print('OUTPUT_DIR =', OUTPUT_DIR)

assert MODEL_PATH.exists(), f"Model not found locally: {MODEL_PATH}"

In [ ]:
# Cell 4 - Import the pipeline from the local repo

import sys

sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from plate_ocr.detect_and_ocr import PlateOCRPipeline

print('Pipeline imported successfully from local source tree.')

In [ ]:
# Cell 5 - Initialize the pipeline with the local model

import torch

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('DEVICE =', DEVICE)

pipeline = PlateOCRPipeline(
    model_path=MODEL_PATH,
    languages=('en',),
    device=DEVICE,
)

print('YOLO + EasyOCR are ready from the local project files.')

In [ ]:
# Cell 6 - Process one uploaded image
# Upload an image from your machine and the notebook will process it using the local model.

from google.colab import files
from IPython.display import Image, display

uploaded = files.upload()
if not uploaded:
    raise ValueError('No file selected.')

uploaded_file_name = next(iter(uploaded))
uploaded_path = PROJECT_ROOT / uploaded_file_name
uploaded_path.write_bytes(uploaded[uploaded_file_name])
IMAGE_PATH.write_bytes(uploaded[uploaded_file_name])

print('Uploaded image saved to:', IMAGE_PATH)
print('Processing:', IMAGE_PATH)

detections, output_path = pipeline.detect_and_read(IMAGE_PATH, conf_threshold=0.25)

print(f"Number of detections: {len(detections)}")
print('Annotated image saved to:', output_path)

display(Image(filename=str(output_path), width=800))

for i, det in enumerate(detections, start=1):
    print(f"Plate {i}: bbox={det['bbox']} confidence={det['confidence']:.3f} texts={det['texts']}")

In [ ]:
# Cell 7 - Process a folder of locally available images
# This version does not use raw data. It only reads images from the local project folder or uploaded files.

if not TEST_IMAGES_DIR.exists():
    raise FileNotFoundError(f"The local test images folder was not found: {TEST_IMAGES_DIR}")

results = pipeline.detect_and_read_directory(TEST_IMAGES_DIR, conf_threshold=0.25)

print(f"Images processed: {len(results)}")
for item in results:
    print(item['image'], '->', len(item['detections']), 'detection(s)')
    for det in item['detections']:
        print('  ', det['bbox'], det.get('texts', []))

In [ ]:
# Cell 8 - Export results to a local CSV file

import pandas as pd

rows = []
for item in results:
    for det in item['detections']:
        rows.append({
            'image': item['image'],
            'bbox': det['bbox'],
            'confidence': det['confidence'],
            'texts': ' | '.join(det.get('texts', [])),
            'output': str(item['output'])
        })

if rows:
    df = pd.DataFrame(rows)
    print(df.head())
    summary_path = PROJECT_ROOT / 'outputs' / 'results_summary.csv'
    df.to_csv(summary_path, index=False)
    print('CSV saved locally to:', summary_path)
else:
    print('No detections found.')